# §8.1 — Run the CME–ODE Whole-Cell Model (4 replicates)

This notebook launches the hybrid **CME–ODE Whole-Cell Model** of JCVI-Syn3A on the **QCB Delta Gateway**, the same way the other CME tutorials run — directly in the Jupyter kernel.

> **Kernel:** select **`LM 2.5 (Python 3.7)`** (top-right of the notebook).

Each of the 4 replicates is an independent run of the driver `programs/WCM_CMEODE_Hook.py`. A short 60-second biological simulation with 4 replicates finishes in roughly **10 minutes**. Output is written into **`output_4replicates/`** inside your cloned copy of the repository.

This 60-second run just confirms the workflow runs. For the actual analysis (full cell cycle), open [`analysis/analysis.ipynb`](analysis/analysis.ipynb), which uses ten pre-run replicates.

## 1. Set parameters and locate the simulation code

The simulation code lives in `programs/` and the input files in `input_data/`, both next to this notebook. The driver reads inputs with paths relative to `programs/`, so we launch from there and write output to a sibling `output_4replicates/` folder in your clone (which is writable, unlike the shared read-only workshop folder).

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# This notebook lives in .../CME/WCM/
WCM_DIR = Path.cwd()
PROGRAMS = WCM_DIR / "programs"
INPUT_DATA = WCM_DIR / "input_data"
OUTPUT_DIR = WCM_DIR / "output_4replicates"   # written into your (writable) cloned repo

assert PROGRAMS.is_dir(), f"programs/ not found at {PROGRAMS}. Open this notebook from the WCM folder."
assert INPUT_DATA.is_dir(), f"input_data/ not found at {INPUT_DATA}."

if not os.access(WCM_DIR, os.W_OK):
    raise PermissionError(
        f"{WCM_DIR} is read-only. Run this notebook from YOUR cloned copy of the "
        "repository, not the shared workshop folder. In a code cell:\n"
        "    !git clone https://github.com/Luthey-Schulten-Lab/SummerSchool_2026.git\n"
        "    %cd SummerSchool_2026/CME/WCM"
    )
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Simulation controls (small defaults so it finishes in ~10 min on 1 GPU) ---
NREP    = 4    # number of independent cell replicates (one MPI rank each)
SIMTIME = 60   # biological seconds to simulate
RESTART = 60   # CME restart interval (s); SIMTIME must be a multiple of this
HOOK    = 1    # CME<->ODE communication (hook) interval (s); RESTART must be a multiple

assert SIMTIME % RESTART == 0, "SIMTIME (-t) must be an integer multiple of RESTART (-rs)"
assert RESTART % HOOK == 0,    "RESTART (-rs) must be an integer multiple of HOOK (-hi)"

PYLM_PATH = "/Software/Lattice_Microbes/src/pylm"
if PYLM_PATH not in sys.path:
    sys.path.insert(0, PYLM_PATH)

print("WCM folder :", WCM_DIR)
print("Programs   :", PROGRAMS)
print("Output dir :", OUTPUT_DIR)
print(f"Replicates : {NREP}  |  sim time: {SIMTIME} s  |  restart: {RESTART} s  |  hook: {HOOK} s")

### Environment check (optional)

Confirms the `LM 2.5 (Python 3.7)` kernel has everything the simulation needs. If anything shows `MISSING`, you have the wrong kernel selected — switch to **`LM 2.5 (Python 3.7)`** and re-run.

In [ ]:
print("Python:", sys.executable)
for mod in ["pyLM", "odecell", "mpi4py", "Bio", "numpy", "pandas"]:
    try:
        __import__(mod)
        print(f"  {mod:10s} OK")
    except Exception as e:
        print(f"  {mod:10s} MISSING -> {e}")
print("mpirun:", shutil.which("mpirun") or "NOT FOUND")

## 2. Run the replicates

Run the `NREP` replicates and collect their output. Each replicate redirects its console output to `log_<i>.txt` inside `output_4replicates/`; the cell below prints a line as each replicate starts and finishes.

In [ ]:
env = dict(os.environ, HYDRA_BOOTSTRAP="fork", HDF5_USE_FILE_LOCKING="FALSE")

# Work around "PMIX ERROR: NO-PERMISSIONS in dstore_base.c" in the Gateway
# container: use PMIx's in-memory 'hash' data store instead of the shared-memory
# one, and point PMIx/MPI at a temp dir we know is writable.
TMP_DIR = OUTPUT_DIR / "tmp"
TMP_DIR.mkdir(exist_ok=True)
env["TMPDIR"] = str(TMP_DIR)
env["PMIX_MCA_gds"] = "hash"
env["PMIX_MCA_psec"] = "native"

env["PYTHONPATH"] = PYLM_PATH + ((os.pathsep + env["PYTHONPATH"]) if env.get("PYTHONPATH") else "")

# The driver names every output by MPI rank, so rename to the replicate index
# after each run.
cmd = [
    "mpirun", "-np", "1",
    sys.executable, "./WCM_CMEODE_Hook.py",
    "-st", "cme-ode",
    "-t",  str(SIMTIME),
    "-rs", str(RESTART),
    "-hi", str(HOOK),
    "-f",  str(OUTPUT_DIR),
]

failures = []
for i in range(1, NREP + 1):
    print(f"=== Replicate {i}/{NREP} ===")
    print("Running:", " ".join(cmd), "\n")
    proc = subprocess.run(cmd, cwd=str(PROGRAMS), env=env)
    print("exit code:", proc.returncode)
    if proc.returncode != 0:
        failures.append(i)
        print(f"  Replicate {i} failed — see {OUTPUT_DIR}/log_1.txt")
        continue
    # rename this run's rank-1 output to the replicate index
    for stem, ext in [("log", "txt"), ("counts", "csv"), ("SA", "csv"), ("Flux", "csv")]:
        src = OUTPUT_DIR / f"{stem}_1.{ext}"
        if src.is_file():
            src.rename(OUTPUT_DIR / f"{stem}_{i}.{ext}")
    print(f"  Replicate {i} done -> counts_{i}.csv, SA_{i}.csv, Flux_{i}.csv, log_{i}.txt\n")

if failures:
    print("\nFailed replicates:", failures, "— check the log files in", OUTPUT_DIR)
else:
    print(f"\nAll {NREP} replicates finished.")

## 3. Check the output

Each replicate `i` produces `counts_i.csv` (species counts), `SA_i.csv` (surface area & volume), `Flux_i.csv` (ODE reaction fluxes) and `log_i.txt`.

In [ ]:
print("Files in", OUTPUT_DIR, ":")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name:22s} {f.stat().st_size/1e3:8.1f} KB")

log1 = OUTPUT_DIR / "log_1.txt"
if log1.is_file():
    print("\n--- tail of log_1.txt ---")
    print("\n".join(log1.read_text().splitlines()[-25:]))

Done — this confirms the CME–ODE workflow runs end to end. A 60-second run is far too short to show cell-cycle behavior, so for the analysis we use ten pre-run, full-cycle replicates: open [`analysis/analysis.ipynb`](analysis/analysis.ipynb) (select the `LM 2.5 (Python 3.7)` kernel, Run All) to reproduce the figures in the [README](README.md#6-analysis-and-discussion).

Want a longer run or more replicates? Increase `SIMTIME` (and `NREP`) in section 1 and re-run — keep `SIMTIME` a multiple of `RESTART`, and `RESTART` a multiple of `HOOK`.